# aloevera — `avr.plot()` Demo

`avr.plot(df, kind, **kwargs)` is a thin wrapper around `plotly.express` that applies
a consistent default layout (`plotly_white` template, 400 × 700, centred title).

```python
avr.plot(df, 'line', x='day', y='price')   # direct call
df.pipe(avr.plot, 'line', x='day', y='price')  # pipe style
```

- Pass any `plotly.express` function name as `kind`
- Polars DataFrames (and LazyFrames) are converted to pandas automatically
- Override any `DEFAULT_LAYOUT` key per-call, or mutate `avr.DEFAULT_LAYOUT` globally

In [1]:
import aloevera as avr
import polars as pl
import numpy as np

print(f"aloevera {avr.__version__}")
print("DEFAULT_LAYOUT:", avr.DEFAULT_LAYOUT)

aloevera 0.1.0
DEFAULT_LAYOUT: {'template': 'plotly_white', 'height': 400, 'width': 700, 'title_x': 0.5, 'color_continuous_scale': 'Spectral', 'xaxis_showline': True, 'xaxis_linewidth': 1, 'xaxis_linecolor': '#888888', 'xaxis_mirror': True, 'yaxis_showline': True, 'yaxis_linewidth': 1, 'yaxis_linecolor': '#888888', 'yaxis_mirror': True}


## 1. Sample data (Polars)

In [3]:
rng = np.random.default_rng(42)

months = ["Jan","Feb","Mar","Apr","May","Jun",
          "Jul","Aug","Sep","Oct","Nov","Dec"]

# Monthly sales
df_sales = pl.DataFrame({
    "month":     months,
    "revenue":   rng.integers(50_000, 150_000, 12).tolist(),
    "expenses":  rng.integers(30_000, 100_000, 12).tolist(),
    "profit":    rng.integers(10_000,  60_000, 12).tolist(),
    "customers": rng.integers(200, 800, 12).tolist(),
})

# 90-day time series
df_ts = pl.DataFrame({
    "day":    list(range(1, 91)),
    "price":  np.cumsum(rng.standard_normal(90) * 2 + 0.1).tolist(),
    "volume": rng.integers(100, 1000, 90).tolist(),
})

# Distribution data — three groups
df_dist = pl.DataFrame({
    "value": np.concatenate([
        rng.normal(50, 10, 200),
        rng.normal(65, 12, 200),
        rng.normal(45,  8, 200),
    ]).tolist(),
    "group": ["A"] * 200 + ["B"] * 200 + ["C"] * 200,
})

# Product category performance
df_cat = pl.DataFrame({
    "category": ["Electronics", "Clothing", "Food", "Books", "Sports", "Home"],
    "q1": rng.integers(10_000, 80_000, 6).tolist(),
    "q2": rng.integers(10_000, 80_000, 6).tolist(),
    "q3": rng.integers(10_000, 80_000, 6).tolist(),
    "q4": rng.integers(10_000, 80_000, 6).tolist(),
})

print("df_sales:",  df_sales.shape)
print("df_ts:",     df_ts.shape)
print("df_dist:",   df_dist.shape)
print("df_cat:",    df_cat.shape)
df_sales.head()

df_sales: (12, 5)
df_ts: (90, 3)
df_dist: (600, 2)
df_cat: (6, 5)


month,revenue,expenses,profit,customers
str,i64,i64,i64,i64
"""Jan""",58925,81502,49078,714
"""Feb""",127395,83279,42193,696
"""Mar""",115457,80223,30120,366
"""Apr""",93887,85024,51138,578
"""May""",93301,65925,37271,299


## 2. Line chart — `kind='line'`

In [4]:
avr.plot(
    df_ts, 'line',
    x='day', y='price',
    title='90-Day Price Trend',
    labels={'day': 'Day', 'price': 'Price ($)'},
    line_shape='spline',
)

## 3. Bar chart — `kind='bar'`

In [5]:
avr.plot(
    df_sales, 'bar',
    x='month', y=['revenue', 'expenses'],
    title='Monthly Revenue vs Expenses',
    barmode='group',
    labels={'value': 'Amount ($)', 'variable': 'Metric'},
)

## 4. Scatter plot — `kind='scatter'`

In [6]:
avr.plot(
    df_sales, 'scatter',
    x='customers', y='revenue',
    size='profit', text='month',
    title='Customers vs Revenue (bubble = profit)',
    labels={'customers': 'Customers', 'revenue': 'Revenue ($)'},
)

## 5. Histogram — `kind='histogram'`

In [7]:
avr.plot(
    df_dist, 'histogram',
    x='value', color='group',
    barmode='overlay', opacity=0.7,
    nbins=40,
    title='Value Distribution by Group',
)

## 6. Box plot — `kind='box'`

In [8]:
avr.plot(
    df_dist, 'box',
    x='group', y='value',
    color='group',
    title='Value Distribution — Box Plot',
    points='outliers',
)

## 7. Violin plot — `kind='violin'`

In [9]:
avr.plot(
    df_dist, 'violin',
    x='group', y='value',
    color='group', box=True,
    title='Value Distribution — Violin Plot',
)

## 8. Area chart — `kind='area'`

Use Polars `unpivot` to reshape to long form before plotting.

In [10]:
df_area = df_sales.unpivot(
    on=['revenue', 'expenses', 'profit'],
    index='month',
    variable_name='metric',
    value_name='amount',
)

avr.plot(
    df_area, 'area',
    x='month', y='amount', color='metric',
    title='Monthly Financials — Stacked Area',
    labels={'amount': 'Amount ($)', 'metric': 'Metric'},
)

## 9. Pie chart — `kind='pie'`

In [11]:
avr.plot(
    df_cat, 'pie',
    names='category', values='q1',
    title='Q1 Revenue by Category',
    hole=0.3,
)

## 10. Density heatmap — `kind='density_heatmap'`

In [12]:
avr.plot(
    df_dist, 'density_heatmap',
    x='value', y='group',
    title='Value Density Heatmap',
    nbinsx=30,
    color_continuous_scale='Blues',
)

## 11. Strip plot — `kind='strip'`

In [13]:
avr.plot(
    df_dist, 'strip',
    x='group', y='value',
    color='group',
    title='Individual Observations by Group',
)

## 12. Using `pipe` for chained Polars transforms

Because `data_frame` is the first argument, `df.pipe(avr.plot, kind, ...)` works
naturally at the end of a Polars expression chain.

In [14]:
# Filter → sort → plot in one pipeline
(
    df_sales
    .filter(pl.col('revenue') > 70_000)
    .sort('revenue', descending=True)
    .pipe(avr.plot, 'bar', x='month', y='revenue', title='High-Revenue Months (>70k)')
)

In [15]:
# Derive a new column, then plot
(
    df_sales
    .with_columns((pl.col('revenue') - pl.col('expenses')).alias('net'))
    .pipe(avr.plot, 'bar', x='month', y='net',
          title='Monthly Net (Revenue − Expenses)',
          color='net', color_continuous_scale='RdYlGn')
)

In [16]:
# Group-aggregate → pipe into a scatter
(
    df_dist
    .group_by('group')
    .agg(
        pl.col('value').mean().alias('mean'),
        pl.col('value').std().alias('std'),
        pl.col('value').count().alias('n'),
    )
    .sort('group')
    .pipe(avr.plot, 'scatter',
          x='group', y='mean', size='std',
          title='Group Mean ± StdDev (bubble size = std)',
          labels={'mean': 'Mean Value', 'group': 'Group'})
)

## 13. Overriding `DEFAULT_LAYOUT` per-call

Pass any `DEFAULT_LAYOUT` key as a kwarg to override it for that call only.

In [17]:
avr.plot(
    df_sales, 'bar',
    x='month', y='revenue',
    title='Revenue (ggplot2, taller)',
    template='ggplot2',
    height=500,
    width=800,
)

## 14. Combining `avr.plot()` with organizers

In [18]:
avr.tabs(
    titles=['Line', 'Bar', 'Scatter', 'Histogram'],
    contents=[
        avr.plot(df_ts,    'line',      x='day',       y='price',    title='Price Trend'),
        avr.plot(df_sales, 'bar',       x='month',     y='revenue',  title='Monthly Revenue'),
        avr.plot(df_sales, 'scatter',   x='customers', y='revenue',  title='Customers vs Revenue'),
        avr.plot(df_dist,  'histogram', x='value',     color='group',title='Distributions'),
    ],
)

    'data': [{'hovertemplate': 'day=%{x}<br>price=%{y}<extra></ext…

In [19]:
avr.accordion(
    titles=['Box', 'Violin', 'Strip'],
    contents=[
        avr.plot(df_dist, 'box',    x='group', y='value', color='group', title='Box'),
        avr.plot(df_dist, 'violin', x='group', y='value', color='group', title='Violin', box=True),
        avr.plot(df_dist, 'strip',  x='group', y='value', color='group', title='Strip'),
    ],
)

    'data': [{'alignmentgroup': 'True',
              'hover…

In [20]:
avr.dropdown(
    titles=['Area', 'Pie', 'Density Heatmap'],
    contents=[
        avr.plot(df_area, 'area',             x='month', y='amount', color='metric', title='Stacked Area'),
        avr.plot(df_cat,  'pie',              names='category', values='q1', title='Q1 by Category', hole=0.3),
        avr.plot(df_dist, 'density_heatmap',  x='value', y='group', title='Density Heatmap', nbinsx=30),
    ],
)

## 15. Changing the global `DEFAULT_LAYOUT`

In [21]:
avr.DEFAULT_LAYOUT['height'] = 500
avr.DEFAULT_LAYOUT['width']  = 900

df_ts.pipe(avr.plot, 'line', x='day', y='price', title='Wider/Taller Default')

In [22]:
# Restore
avr.DEFAULT_LAYOUT.update({'height': 400, 'width': 700})
print("Restored:", avr.DEFAULT_LAYOUT)

Restored: {'template': 'plotly_white', 'height': 400, 'width': 700, 'title_x': 0.5, 'color_continuous_scale': 'RdBu', 'xaxis_showline': True, 'xaxis_linewidth': 1, 'xaxis_linecolor': '#888888', 'xaxis_mirror': True, 'yaxis_showline': True, 'yaxis_linewidth': 1, 'yaxis_linecolor': '#888888', 'yaxis_mirror': True}
